# CS431 — Chọn ảnh tham chiếu và thích nghi phơi sáng cho Depth Anything 3

Notebook so sánh Combined với các chiến lược chọn tham chiếu của DA3 và đo phần đóng góp của Exposure-Consistency LoRA. Bốn cảnh pilot dùng để huấn luyện và năm cảnh giữ lại dùng để đánh giá.

Mặc định notebook nạp hai checkpoint đã lưu và chạy đánh giá. Đặt `RETRAIN = True` trong phần thiết lập để huấn luyện lại trên bốn cảnh pilot. Cần Kaggle GPU và Internet. Kết quả được ghi vào `/kaggle/working/outputs`.


In [ ]:
import csv
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from PIL import Image

WORK_DIR = Path('/kaggle/working')
OUT_DIR = WORK_DIR / 'outputs'
TMP_DIR = Path(tempfile.gettempdir()) / 'cs431-realx3d'
REPO_DIR = TMP_DIR / 'Depth-Anything-3'
DATA_DIR = TMP_DIR / 'realx3d'
PC_DIR = TMP_DIR / 'pointclouds'

PROJECT_URL = 'https://github.com/ThanhNosaur/DoAnCS431.git'
DA3_REPO_URL = 'https://github.com/ByteDance-Seed/Depth-Anything-3.git'
DA3_COMMIT = '3d835ec1a5802d64a8b8b15f817a1ab54809bfe4'
HF_REALX3D_BASE = 'https://huggingface.co/datasets/ToferFish/RealX3D/resolve/main'
VARYEXP_TAR_URL = f'{HF_REALX3D_BASE}/data_4/data4_varyexp.tar.gz'

TRAIN_SCENES = ['Ujikintoki', 'BlueHawaii', 'Chocolate', 'Cupcake']
HELDOUT_SCENES = ['GearWorks', 'Laboratory', 'MilkCookie', 'Popcorn', 'Sculpture']
ALL_SCENES = TRAIN_SCENES + HELDOUT_SCENES
NUM_VIEWS = 4
EVAL_RES = 504
TRAIN_LORA_RES = 336
SEED = 42
RETRAIN = False

random.seed(SEED)
np.random.seed(SEED)

for d in (OUT_DIR, TMP_DIR, DATA_DIR, PC_DIR):
    d.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(map(str, cmd))}", flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

def locate_project() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if all((candidate / 'cs431' / name).is_file() for name in
               ('run_experiment.py', 'learned_selector.py', 'lora_adapt.py')):
            return candidate
    checkout = TMP_DIR / 'DoAnCS431'
    if not checkout.exists():
        run_cmd(['git', 'clone', '--depth', '1', PROJECT_URL, str(checkout)])
    return checkout

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before running the notebook.')
run_cmd(['nvidia-smi'], check=False)
if not (REPO_DIR / 'src' / 'depth_anything_3' / 'api.py').is_file():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd(['git', 'init'], cwd=REPO_DIR)
    run_cmd(['git', 'fetch', '--depth', '1', DA3_REPO_URL, DA3_COMMIT], cwd=REPO_DIR)
    run_cmd(['git', 'checkout', 'FETCH_HEAD'], cwd=REPO_DIR)

PROJECT_DIR = locate_project()
CHECKPOINT_DIR = PROJECT_DIR / 'results' / 'train_ab'

pkgs = [
    'addict', 'omegaconf', 'einops>=0.8', 'huggingface_hub<1', 'safetensors',
    'evo', 'plyfile', 'trimesh', 'e3nn', 'moviepy==1.0.3', 'imageio', 'pycolmap', 'pillow_heif'
]
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

os.environ['HF_HOME'] = str(TMP_DIR / 'hf_cache')
module_paths = (str(REPO_DIR / 'src'), str(PROJECT_DIR))
for p in module_paths:
    if p not in sys.path:
        sys.path.insert(0, p)

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join(module_paths)
run_cmd([sys.executable, '-m', 'unittest', 'cs431.test_experiment', '-v'], cwd=PROJECT_DIR, env=env)


## 1. Chuẩn bị dữ liệu RealX3D


In [ ]:
def stream_extract_varyexp(target_scenes: list[str]):
    target_set = set(target_scenes)
    extracted_files = 0
    extracted_bytes = 0
    t0 = perf_counter()
    print(f'Streaming {VARYEXP_TAR_URL} for {len(target_scenes)} scenes...', flush=True)
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', VARYEXP_TAR_URL],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                parts = Path(member.name).parts
                if len(parts) < 3:
                    continue
                scene = parts[1] if parts[0] == 'varyexp' else parts[0]
                rel_parts = parts[2:] if parts[0] == 'varyexp' else parts[1:]
                if scene not in target_set or not rel_parts:
                    continue
                top = rel_parts[0]
                is_meta = top in ('transforms_train.json', 'colmap2world.txt')
                is_sparse = top == 'sparse' and Path(member.name).name in ('images.bin', 'images.txt')
                is_train_img = top.startswith('train') and member.name.lower().endswith(('.jpg', '.jpeg', '.png'))
                if '..' in rel_parts or not (is_meta or is_sparse or is_train_img):
                    continue
                dst = DATA_DIR / scene / Path(*rel_parts)
                dst.parent.mkdir(parents=True, exist_ok=True)
                src_f = tar.extractfile(member)
                if src_f is not None:
                    with dst.open('wb') as out_f:
                        shutil.copyfileobj(src_f, out_f)
                    extracted_files += 1
                    extracted_bytes += member.size
                    if extracted_files % 250 == 0:
                        print(f'  extracted {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl exited with status {ret}')
    print(f'Done streaming: {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)

def download_pointcloud(scene: str) -> Path:
    url = f'{HF_REALX3D_BASE}/pointclouds/{scene}.tar.gz'
    scene_pc_dir = PC_DIR / scene
    scene_pc_dir.mkdir(parents=True, exist_ok=True)
    ply_path = None
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', url],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if member.isfile() and Path(member.name).name == 'cull_pointcloud.ply':
                    dst = scene_pc_dir / 'cull_pointcloud.ply'
                    src_f = tar.extractfile(member)
                    if src_f is not None:
                        with dst.open('wb') as out_f:
                            shutil.copyfileobj(src_f, out_f)
                    ply_path = dst
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl failed for {url} (exit {ret})')
    return ply_path

stream_extract_varyexp(ALL_SCENES)


## 2. Bộ chọn ảnh tham chiếu học được dùng để đối chứng


In [ ]:
from plyfile import PlyData
from depth_anything_3.api import DepthAnything3
from cs431.geometry import cloud_metrics, predicted_poses, relative_pose_metrics, to_metric_poses, voxel_sample
from cs431.run_experiment import DA3_BASE, DA3_BASE_REVISION, evaluate_views, load_colmap_views
from cs431.learned_selector import LearnedReferenceScorer, extract_view_features, train_reference_scorer
from cs431.lora_adapt import attach_lora_to_da3, exposure_consistency_loss, load_lora_weights, save_lora_weights, set_lora_enabled

torch.manual_seed(SEED)
device = torch.device('cuda:0')
print('Loading DA3-Base on', torch.cuda.get_device_name(0), flush=True)
model = DepthAnything3.from_pretrained(DA3_BASE, revision=DA3_BASE_REVISION).to(device).eval()

def load_gt_cloud(ply_path: Path) -> np.ndarray:
    vertices = PlyData.read(str(ply_path))['vertex'].data
    xyz = np.column_stack([vertices[a] for a in 'xyz']).astype(float)
    return voxel_sample(xyz)

def _exposure_sort_key(folder_name: str):
    m = re.search(r'(\d+)', folder_name)
    return (0, int(m.group(1))) if m else (1, folder_name)

def get_scene_view_sets(scene_dir: Path, offsets=(0, 2, 4, 6), num_views: int = NUM_VIEWS):
    frames = json.loads((scene_dir / 'transforms_train.json').read_text(encoding='utf-8'))['frames']
    train_dirs = sorted(
        [p for p in scene_dir.iterdir() if p.is_dir() and p.name.startswith('train')],
        key=lambda p: _exposure_sort_key(p.name),
    )
    iso_dirs = [p for p in train_dirs if re.search(r'\d+', p.name)]
    sets = []
    for k, off in enumerate(offsets):
        start_idx = min(off, max(0, len(frames) - num_views))
        end_idx = min(len(frames) - 1, max(start_idx + num_views - 1, len(frames) - 1 - k))
        idxs = np.linspace(start_idx, end_idx, num_views, dtype=int)
        sampled = [frames[i] for i in idxs]
        names = [Path(f['file_path']).name for f in sampled]
        default_paths = [scene_dir / f['file_path'] for f in sampled]
        if all(p.is_file() for p in default_paths):
            sets.append((f'default_off{off}', default_paths))
        complete_iso = [d for d in iso_dirs if all((d / n).is_file() for n in names)]
        if len(complete_iso) >= 2:
            picks = np.linspace(0, len(complete_iso) - 1, num_views, dtype=int)
            # Rotate exposure assignment per offset for diversity
            picks = np.roll(picks, off % num_views)
            sets.append((f'vary_off{off}', [complete_iso[pi] / n for pi, n in zip(picks, names)]))
            mid_dir = complete_iso[len(complete_iso) // 2]
            sets.append((f'uniform_off{off}', [mid_dir / n for n in names]))
    return sets


selector_path = CHECKPOINT_DIR / 'learned_selector_A.pt'
if RETRAIN:
    t_a0 = perf_counter()
    train_samples = []
    for scene in TRAIN_SCENES:
        scene_dir = DATA_DIR / scene
        ply_path = download_pointcloud(scene)
        gt_points = load_gt_cloud(ply_path)
        c2w_mat = np.loadtxt(scene_dir / 'colmap2world.txt')
        view_sets = get_scene_view_sets(scene_dir, offsets=(0, 2, 4, 6))
        print(f'[Dir A] Collecting {len(view_sets)} training sets from {scene}...', flush=True)
        for tag, img_paths in view_sets:
            paths, gt_poses = load_colmap_views(scene_dir / 'sparse' / '0', img_paths)
            gt_metric = to_metric_poses(gt_poses, c2w_mat)
            imgs = [Image.open(p).convert('RGB') for p in paths]
            feats, _ = extract_view_features(imgs, model=model, process_res=336)
            f1s = []
            for ref_i in range(NUM_VIEWS):
                order = [ref_i] + [j for j in range(NUM_VIEWS) if j != ref_i]
                ordered = [imgs[j] for j in order]
                pred = model.inference(ordered, ref_view_strategy='first', process_res=EVAL_RES)
                m = cloud_metrics(pred, gt_metric[order], gt_points)
                f1s.append(float(m['f1']) if m.get('f1') is not None else 0.0)
            train_samples.append((feats, np.asarray(f1s, dtype=np.float32)))
        shutil.rmtree(PC_DIR / scene, ignore_errors=True)

    scorer_A, loss_hist_A = train_reference_scorer(train_samples, epochs=300, lr=3e-3, weight_decay=2e-2, tau=0.04, seed=SEED)
    torch.save(scorer_A.state_dict(), OUT_DIR / 'learned_selector_A.pt')
    print(f'[Dir A] Trained on {len(train_samples)} sets in {perf_counter() - t_a0:.1f}s | loss: {loss_hist_A[0]:.4f} -> {loss_hist_A[-1]:.4f}', flush=True)
    selector_path = OUT_DIR / 'learned_selector_A.pt'
else:
    scorer_A = LearnedReferenceScorer()
    scorer_A.load_state_dict(torch.load(selector_path, map_location='cpu', weights_only=True))
    scorer_A.eval()


## 3. Exposure-Consistency LoRA

Khi huấn luyện lại, DA3 gốc nhận ảnh phơi sáng đồng đều và cung cấp mục tiêu cho DA3 gắn LoRA khi nhận ảnh cùng góc nhìn với mức phơi sáng khác.


In [ ]:
t_b0 = perf_counter()
lora_params = attach_lora_to_da3(model, rank=4, alpha=8.0, start_block=6)
n_lora = sum(p.numel() for p in lora_params)
n_total = sum(p.numel() for p in model.parameters())
print(f'[Dir B] Attached LoRA: {n_lora:,} trainable / {n_total:,} total ({100.0 * n_lora / n_total:.3f}%)', flush=True)

lora_path = CHECKPOINT_DIR / 'da3_lora_r4.pt'
if RETRAIN:
    # Build paired (varyexp, clean) image path pools from the 4 train scenes
    train_scene_pools = []
    for scene in TRAIN_SCENES:
        sdir = DATA_DIR / scene
        frames = json.loads((sdir / 'transforms_train.json').read_text(encoding='utf-8'))['frames']
        names = [Path(f['file_path']).name for f in frames]
        iso_dirs = sorted(
            [p for p in sdir.iterdir() if p.is_dir() and re.search(r'\d+', p.name)],
            key=lambda p: _exposure_sort_key(p.name),
        )
        clean_dir = iso_dirs[len(iso_dirs) // 2]
        all_vary_dirs = iso_dirs + ([sdir / 'train'] if (sdir / 'train').is_dir() else [])
        train_scene_pools.append((scene, names, clean_dir, all_vary_dirs))

    LORA_STEPS = 160
    GRAD_ACCUM = 4
    opt_b = torch.optim.AdamW(lora_params, lr=1e-4, weight_decay=1e-2)
    sched_b = torch.optim.lr_scheduler.CosineAnnealingLR(opt_b, T_max=LORA_STEPS // GRAD_ACCUM)
    scaler = torch.amp.GradScaler('cuda')

    model.eval()  # Keep frozen BN/Dropout in eval mode while training LoRA weights
    opt_b.zero_grad()
    running_losses = []
    for step in range(1, LORA_STEPS + 1):
        scene, names, clean_dir, vary_dirs = random.choice(train_scene_pools)
        # Sample 4 sorted frame indices spanning the trajectory
        idxs = sorted(random.sample(range(len(names)), NUM_VIEWS))
        chosen_names = [names[i] for i in idxs]
        clean_imgs = [Image.open(clean_dir / n).convert('RGB') for n in chosen_names]

        if random.random() < 0.20:
            # Identity regularization on clean views
            vary_imgs = clean_imgs
        else:
            vary_imgs = [Image.open(random.choice(vary_dirs) / n).convert('RGB') for n in chosen_names]

        # Teacher pass (frozen base DA3, LoRA disabled)
        set_lora_enabled(model, False)
        with torch.no_grad():
            c_cpu, _, _ = model._preprocess_inputs(clean_imgs, None, None, process_res=TRAIN_LORA_RES)
            c_t, _, _ = model._prepare_model_inputs(c_cpu, None, None)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                pred_t = model.model(c_t, ref_view_strategy='first')

        # Student pass (LoRA enabled)
        set_lora_enabled(model, True)
        v_cpu, _, _ = model._preprocess_inputs(vary_imgs, None, None, process_res=TRAIN_LORA_RES)
        v_t, _, _ = model._prepare_model_inputs(v_cpu, None, None)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred_s = model.model(v_t, ref_view_strategy='first')
            loss, stats = exposure_consistency_loss(pred_s, pred_t)
            loss_scaled = loss / GRAD_ACCUM

        scaler.scale(loss_scaled).backward()
        running_losses.append(stats['total'])
        if step % GRAD_ACCUM == 0:
            scaler.unscale_(opt_b)
            torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
            scaler.step(opt_b)
            scaler.update()
            opt_b.zero_grad()
            sched_b.step()
        if step % 40 == 0:
            peak_gb = torch.cuda.max_memory_allocated(device) / 1024**3
            print(f'  step {step:3d}/{LORA_STEPS} | mean_loss={np.mean(running_losses[-40:]):.4f} | silog={stats["silog"]:.4f} | grad={stats["grad"]:.4f} | pose={stats["pose"]:.4f} | peak_vram={peak_gb:.2f}GB', flush=True)

    save_lora_weights(model, OUT_DIR / 'da3_lora_r4.pt')
    print(f'[Dir B] LoRA training complete in {perf_counter() - t_b0:.1f}s', flush=True)
    torch.cuda.empty_cache()
    lora_path = OUT_DIR / 'da3_lora_r4.pt'
else:
    load_lora_weights(model, lora_path)


## 4. Đánh giá và lưu kết quả

Thử từng ảnh làm tham chiếu trên cả chín cảnh. Đánh giá các tổ hợp có huấn luyện trên năm cảnh giữ lại. Với các cảnh giữ lại, luân phiên bốn mức phơi sáng giữa cùng bốn góc nhìn để kiểm tra độ nhạy với cách gán phơi sáng.


In [ ]:
def build_eval_conditions(scene_dir: Path, include_rotations: bool):
    frames = json.loads((scene_dir / 'transforms_train.json').read_text(encoding='utf-8'))['frames']
    idxs = np.linspace(0, len(frames) - 1, NUM_VIEWS, dtype=int)
    sampled = [frames[i] for i in idxs]
    names = [Path(f['file_path']).name for f in sampled]
    iso_dirs = sorted(
        [p for p in scene_dir.glob('train*') if p.is_dir() and re.search(r'\d+', p.name)],
        key=lambda p: _exposure_sort_key(p.name),
    )
    conds = {'default_json': [scene_dir / f['file_path'] for f in sampled]}
    complete_iso = [d for d in iso_dirs if all((d / n).is_file() for n in names)]
    if len(complete_iso) < 2 or not all(p.is_file() for p in conds['default_json']):
        raise ValueError(f'Incomplete exposure conditions for {scene_dir.name}')
    picks = np.linspace(0, len(complete_iso) - 1, NUM_VIEWS, dtype=int)
    conds['varyexp_mixed'] = [complete_iso[pi] / n for pi, n in zip(picks, names)]
    mid_dir = complete_iso[len(complete_iso) // 2]
    conds['uniform_mid'] = [mid_dir / n for n in names]
    if include_rotations:
        if len(complete_iso) < NUM_VIEWS:
            raise ValueError(f'Need {NUM_VIEWS} distinct exposures for {scene_dir.name}')
        for shift in range(1, NUM_VIEWS):
            conds[f'varyexp_roll{shift}'] = [
                complete_iso[pi] / n for pi, n in zip(np.roll(picks, shift), names)
            ]
    return conds

@torch.inference_mode()
def eval_one(images, gt_poses, gt_metric, gt_points, lora_on: bool, strategy: str, ref_idx: int | None = None):
    set_lora_enabled(model, lora_on)
    if ref_idx is not None:
        order = [ref_idx] + [i for i in range(len(images)) if i != ref_idx]
        ordered = [images[i] for i in order]
        pred = model.inference(ordered, ref_view_strategy='first', process_res=EVAL_RES)
    else:
        order = list(range(len(images)))
        pred = model.inference(images, ref_view_strategy=strategy, process_res=EVAL_RES)
    poses = predicted_poses(pred.extrinsics)
    rel = relative_pose_metrics(poses, gt_poses[order])
    cm = cloud_metrics(pred, gt_metric[order], gt_points)
    return {**rel, **cm, 'predicted_c2w': poses[np.argsort(order)].tolist()}

rows = []
rotation_rows = []
baseline_rows = {'pilot': [], 'heldout': []}
for scene in ALL_SCENES:
    split = 'pilot' if scene in TRAIN_SCENES else 'heldout'
    split_dir = OUT_DIR / split
    split_dir.mkdir(exist_ok=True)
    scene_dir = DATA_DIR / scene
    ply_path = download_pointcloud(scene)
    gt_points = load_gt_cloud(ply_path)
    c2w_mat = np.loadtxt(scene_dir / 'colmap2world.txt')
    conds = build_eval_conditions(scene_dir, include_rotations=(split == 'heldout'))
    extra_dir = OUT_DIR / 'exposure_rotations'
    extra_dir.mkdir(exist_ok=True)

    for cond_name, img_paths in conds.items():
        paths, gt_poses = load_colmap_views(scene_dir / 'sparse' / '0', img_paths)
        gt_metric = to_metric_poses(gt_poses, c2w_mat)
        imgs = [Image.open(p).convert('RGB') for p in paths]

        set_lora_enabled(model, False)

        def infer(ordered, strategy, order):
            torch.cuda.synchronize(device)
            torch.cuda.reset_peak_memory_stats(device)
            t0 = perf_counter()
            prediction = model.inference(ordered, ref_view_strategy=strategy, process_res=EVAL_RES)
            torch.cuda.synchronize(device)
            res = {
                'inference_seconds': perf_counter() - t0,
                'peak_cuda_mib': torch.cuda.max_memory_allocated(device) / 1024**2,
            }
            t1 = perf_counter()
            poses = predicted_poses(prediction.extrinsics)
            res.update(relative_pose_metrics(poses, gt_poses[order]))
            res['predicted_c2w'] = poses[np.argsort(order)].tolist()
            res.update(cloud_metrics(prediction, gt_metric[order], gt_points))
            res['metric_seconds'] = perf_counter() - t1
            return res

        report = {
            'model': DA3_BASE,
            'model_revision': DA3_BASE_REVISION,
            'coordinate_mode': 'colmap_sparse + c2w',
            'device': str(device),
            'resolution': EVAL_RES,
            'images': [str(p) for p in paths],
            **evaluate_views(imgs, infer, sweep=True),
        }
        is_rotation = cond_name.startswith('varyexp_roll')
        out_json = (extra_dir if is_rotation else split_dir) / f'{scene}_{cond_name}.json'
        report['gt_c2w'] = gt_poses.tolist()
        if cond_name.startswith('varyexp_') and split == 'heldout':
            lora_result = eval_one(imgs, gt_poses, gt_metric, gt_points, True, 'first',
                                   report['runs']['combined']['reference_index'])
            report['runs']['lora_combined'] = lora_result
            rotation_row = {'scene': scene, 'condition': cond_name,
                            'exposures': '/'.join(p.parent.name for p in paths)}
            for key, run in (
                ('first', report['sweep'][0]),
                *((name, report['runs'][name]) for name in
                  ('saddle_balanced', 'overlap', 'combined', 'lora_combined')),
            ):
                rotation_row[f'{key}_f1'] = run['f1']
                rotation_row[f'{key}_auc10'] = run['auc10']
            rotation_rows.append(rotation_row)
        if is_rotation:
            out_json.write_text(json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')
            print(scene, cond_name, rotation_rows[-1]['combined_f1'], flush=True)
            continue

        sweep_f1s = [r['f1'] for r in report['sweep'] if r.get('f1') is not None]
        sweep_aucs = [r['auc10'] for r in report['sweep']]
        row = {
            'scene': scene,
            'condition': cond_name,
            'selection_s': round(report['selection_seconds'], 4),
        }
        for m in ('saddle_balanced', 'exposure', 'exposure_bridge', 'overlap', 'combined'):
            r = report['runs'][m]
            row[f'{m}_ref'] = r.get('reference_index')
            row[f'{m}_f1'] = round(r['f1'], 4) if r.get('f1') is not None else None
            row[f'{m}_auc10'] = round(r['auc10'], 4)
            row[f'{m}_med_m'] = round(r['median_pred_to_gt_m'], 4) if r.get('median_pred_to_gt_m') is not None else None
            row[f'{m}_s'] = round(r['inference_seconds'], 3)
        row['first_f1'] = round(report['sweep'][0]['f1'], 4)
        row['first_auc10'] = round(report['sweep'][0]['auc10'], 4)
        oracle = report.get('oracle_f1')
        row['oracle_f1'] = round(oracle['f1'], 4) if oracle else None
        row['span_f1'] = round(max(sweep_f1s) - min(sweep_f1s), 4) if sweep_f1s else None
        row['oracle_auc10'] = round(max(sweep_aucs), 4)
        row['span_auc10'] = round(max(sweep_aucs) - min(sweep_aucs), 4)
        baseline_rows[split].append(row)
        if split == 'pilot':
            out_json.write_text(json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')
            continue

        set_lora_enabled(model, False)
        feats, _ = extract_view_features(imgs, model=model, process_res=336)
        ref_heur = report['runs']['combined']['reference_index']
        with torch.no_grad():
            ref_learned = int(torch.argmax(scorer_A(torch.from_numpy(feats).float())).item())

        r_base_def = report['runs']['saddle_balanced']
        r_base_comb = report['runs']['combined']
        r_base_A = report['sweep'][ref_learned]

        r_lora_def = eval_one(imgs, gt_poses, gt_metric, gt_points, True, 'saddle_balanced', None)
        r_lora_comb = report['runs'].get('lora_combined')
        if r_lora_comb is None:
            r_lora_comb = eval_one(imgs, gt_poses, gt_metric, gt_points, True, 'first', ref_heur)
        r_lora_AB = r_lora_comb if ref_learned == ref_heur else eval_one(imgs, gt_poses, gt_metric, gt_points, True, 'first', ref_learned)

        row = {
            'scene': scene,
            'condition': cond_name,
            'ref_heur': ref_heur,
            'ref_learned': ref_learned,
            'base_def_f1': round(r_base_def['f1'], 4), 'base_def_auc10': round(r_base_def['auc10'], 4),
            'base_comb_f1': round(r_base_comb['f1'], 4), 'base_comb_auc10': round(r_base_comb['auc10'], 4),
            'base_A_f1': round(r_base_A['f1'], 4), 'base_A_auc10': round(r_base_A['auc10'], 4),
            'lora_def_f1': round(r_lora_def['f1'], 4), 'lora_def_auc10': round(r_lora_def['auc10'], 4),
            'lora_comb_f1': round(r_lora_comb['f1'], 4), 'lora_comb_auc10': round(r_lora_comb['auc10'], 4),
            'lora_AB_f1': round(r_lora_AB['f1'], 4), 'lora_AB_auc10': round(r_lora_AB['auc10'], 4),
        }
        report['runs'].update(base_learned=r_base_A, lora_default=r_lora_def,
                              lora_combined=r_lora_comb, lora_learned=r_lora_AB)
        out_json.write_text(json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')
        rows.append(row)
        print(
            f"{scene:11s} {cond_name:14s} | F1: BaseDef={row['base_def_f1']} BaseComb={row['base_comb_f1']} "
            f"Base+A={row['base_A_f1']} LoRA+Def={row['lora_def_f1']} LoRA+Comb={row['lora_comb_f1']} LoRA+AB={row['lora_AB_f1']}",
            flush=True,
        )
    shutil.rmtree(PC_DIR / scene, ignore_errors=True)

assert len(rotation_rows) == len(HELDOUT_SCENES) * NUM_VIEWS
assert len(rows) == len(HELDOUT_SCENES) * 3
for split, summary in baseline_rows.items():
    with (OUT_DIR / split / 'summary.csv').open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)

with (extra_dir / 'summary.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(rotation_rows[0]))
    writer.writeheader()
    writer.writerows(rotation_rows)

with (OUT_DIR / 'summary_ab.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

md = [
    '# CS431 Training Experiment: Direction A (Learned Selector) & Direction B (LoRA)',
    '',
    '## Mean F1 @ 5cm on 5 Held-out Scenes',
    '| Condition | Base+Def | Base+Combined | Base+A (Learned) | LoRA+Def (B) | LoRA+Combined | LoRA+AB |',
    '| --- | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for cond in ('default_json', 'varyexp_mixed', 'uniform_mid'):
    sub = [r for r in rows if r['condition'] == cond]
    m = lambda k: round(float(np.mean([r[k] for r in sub])), 4)
    md.append(f"| {cond} | {m('base_def_f1')} | {m('base_comb_f1')} | {m('base_A_f1')} | {m('lora_def_f1')} | {m('lora_comb_f1')} | {m('lora_AB_f1')} |")

md += ['', '## Mean AUC@10 on 5 Held-out Scenes', '| Condition | Base+Def | Base+Combined | Base+A (Learned) | LoRA+Def (B) | LoRA+Combined | LoRA+AB |', '| --- | ---: | ---: | ---: | ---: | ---: | ---: |']
for cond in ('default_json', 'varyexp_mixed', 'uniform_mid'):
    sub = [r for r in rows if r['condition'] == cond]
    m = lambda k: round(float(np.mean([r[k] for r in sub])), 4)
    md.append(f"| {cond} | {m('base_def_auc10')} | {m('base_comb_auc10')} | {m('base_A_auc10')} | {m('lora_def_auc10')} | {m('lora_comb_auc10')} | {m('lora_AB_auc10')} |")

md_text = '\n'.join(md) + '\n'
(OUT_DIR / 'summary_ab.md').write_text(md_text, encoding='utf-8')
print('\n' + md_text)

provenance = {
    'pose_metric': 'mean AUC@10 of both directed poses per unordered camera pair',
    'model': DA3_BASE,
    'model_revision': DA3_BASE_REVISION,
    'upstream_commit': DA3_COMMIT,
    'resolution': EVAL_RES,
    'gpu': torch.cuda.get_device_name(device),
    'torch': torch.__version__,
    'retrained': RETRAIN,
    'checkpoint_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                          for p in (selector_path, lora_path)},
    'source_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                      for p in sorted((PROJECT_DIR / 'cs431').glob('*.py'))},
}
(OUT_DIR / 'evaluation_provenance.json').write_text(
    json.dumps(provenance, indent=2) + '\n', encoding='utf-8'
)
